# Linear Regression

---

## Introduction

In the Perceptron notebook, we built a model that answered a **yes or no question** — is this dog breed long-lived or short-lived? That type of problem is called **classification**.

But what if the answer isn't yes or no? What if we want to predict an actual **number**?

That's a **regression** problem. In regression, the target labels $y^{(i)}$ are real-valued numbers, not categories. The goal is to find a hypothesis $h: \mathcal{X} \rightarrow \mathcal{Y}$ that approximates the true (unknown) relationship between inputs and outputs.

**Linear Regression** is the simplest form of regression — it assumes that the relationship between the input and output is roughly linear, and tries to find the best straight line through the data:

$$\hat{y} = w \cdot x + b$$

Where $w$ is the slope (weight), $b$ is the intercept (bias), $x$ is the input, and $\hat{y}$ is the predicted output. The model learns $w$ and $b$ by minimizing its prediction error over the training data using **gradient descent**.

In this notebook we will use Linear Regression to answer one question:

> *Can we predict a state's winter bee colony loss percentage based on how many colonies it has?*

**Dataset:** Bee Colony Loss — compiled by the **Bee Informed Partnership (BIP)** and the **USDA Agricultural Statistics Service**  
**Source:** BIP annual colony loss reports (2010–2021)

**Feature used:** number of colonies per state ($x$)  
**Target:** winter colony loss percentage ($y$)

In [ ]:
# ── Standard libraries ───────────────────────────────────────────────────────
import numpy as np        # numerical computing — arrays, math operations
import pandas as pd       # data manipulation — loading and cleaning our dataset
import matplotlib.pyplot as plt  # plotting — visualizing our data and results
import seaborn as sns     # prettier plots built on top of matplotlib

# ── Our custom LinearRegression from the package ─────────────────────────────
import sys
sys.path.insert(0, '../../../Python Package')
from final_ml.supervised_learning.linear_regression import LinearRegression

# ── Plot styling ──────────────────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('All libraries loaded successfully!')

In [ ]:
# Load the dataset
df = pd.read_csv('data/Bee_Colony_Loss.csv')

# First look at the data
print('Dataset shape:', df.shape)
print()
df.head()

## Step 1 — Understanding Gradient Descent

Before we can train a Linear Regression model, we need to understand **how** it learns. The mechanism is called **gradient descent** — the same idea that underlies almost every modern machine learning algorithm.

### The Core Idea

Suppose we want to minimize a simple function of one variable:

$$f(w) = (w - 2)^2 + 1$$

This function has a clear minimum at $w = 2$. But suppose we didn't already know that. How would we find it?

**Gradient descent** gives us a strategy: start with a guess, look at the slope of the function at that point (the derivative), and take a small step in the *downhill* direction. Repeat until you reach the bottom.

The update rule for a single-variable function is:

$$w_{n+1} = w_n - \alpha \cdot f'(w_n)$$

Where $\alpha$ (alpha) is the **learning rate** — it controls how big each step is.

In [ ]:
# Define our simple test function and its derivative
def f(w):
    return (w - 2)**2 + 1

def deriv(w):
    return 2*(w - 2)

# Plot the function
domain = np.linspace(-2, 6, 50)

plt.figure(figsize=(10, 6))
plt.plot(domain, f(domain), label='$f(w) = (w-2)^2 + 1$')
plt.xlabel('w (weight)', fontsize=15)
plt.ylabel('f(w)', fontsize=15)
plt.legend(fontsize=13)
plt.title('Function We Want to Minimize', fontsize=16)
plt.show()

In [ ]:
# Helper function to draw the tangent line at a given point on the curve
# The tangent line shows the slope at that point — which tells us which way is downhill
def draw_tangent(w_i, function, derivative, label, color, show_line=True):
    # Tangent line formula: y = slope*(x - x1) + y1
    def tangent(w):
        return derivative(w_i) * (w - w_i) + function(w_i)

    wrange = np.linspace(w_i - 1, w_i + 1, 10)
    if show_line:
        plt.plot(wrange, tangent(wrange), '--', color='red', linewidth=1.5, label='tangent line')
    plt.scatter([w_i], [function(w_i)], color=color, zorder=5, label=label)

# Start with an initial guess of w_0 = 5
w_0 = 5.0
alpha = 0.8

# One step of gradient descent
w_1 = w_0 - alpha * deriv(w_0)

print(f'w_0 = {w_0}  →  f(w_0) = {f(w_0)}')
print(f'w_1 = {w_1}  →  f(w_1) = {f(w_1)}')
print(f'We moved from {w_0} to {w_1} — closer to the minimum at w=2!')

plt.figure(figsize=(10, 6))
plt.plot(domain, f(domain), label='$f(w) = (w-2)^2 + 1$')
draw_tangent(w_0, f, deriv, label=f'$w_0 = {w_0}$ (initial guess)', color='magenta')
draw_tangent(w_1, f, deriv, label=f'$w_1 = {w_1:.2f}$ (after 1 step)', color='green', show_line=False)
plt.xlabel('w (weight)', fontsize=15)
plt.ylabel('f(w)', fontsize=15)
plt.legend(fontsize=12)
plt.title('One Step of Gradient Descent', fontsize=16)
plt.show()

In [ ]:
# Now let's automate this — run gradient descent until the derivative is near zero
# (near zero slope means we're at or near the minimum)

def derivative_descent(derivative, alpha=0.8, w_0=5.0, max_iter=1000):
    W = [w_0]   # keep track of every w value we visit
    i = 0
    while abs(derivative(W[-1])) > 0.001 and i < max_iter:
        w_new = W[-1] - alpha * derivative(W[-1])
        W.append(w_new)
        i += 1
    return np.array(W)

W = derivative_descent(deriv)

# Print each step
for i, w in enumerate(W):
    print(f'w_{i} = {round(w, 4)}  |  deriv(w_{i}) = {round(deriv(w), 4)}')

# Plot the descent path on the function
plt.figure(figsize=(10, 6))
plt.plot(domain, f(domain), label='$f(w) = (w-2)^2 + 1$')
plt.scatter(W, f(W), color='magenta', zorder=5, label='Gradient descent steps')
plt.xlabel('w (weight)', fontsize=15)
plt.ylabel('f(w)', fontsize=15)
plt.legend(fontsize=12)
plt.title(f'Full Gradient Descent  (alpha={0.8}, {len(W)} steps)', fontsize=16)
plt.show()

In [ ]:
# How does the choice of learning rate affect convergence?
# Too small: takes many steps. Too large: overshoots or diverges.

alphas = [0.1, 0.5, 0.8, 1.0]

fig, axs = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Effect of Learning Rate on Gradient Descent', fontsize=16)

for ax, alpha in zip(axs.flat, alphas):
    W = derivative_descent(deriv, alpha=alpha)
    ax.plot(domain, f(domain))
    ax.scatter(W, f(W), color='magenta')
    ax.set_title(f'alpha = {alpha},  steps = {len(W)}', fontsize=13)
    ax.set_xlabel('w')
    ax.set_ylabel('f(w)')

plt.tight_layout()
plt.show()

## Step 2 — Understanding the Data

We have state-level observations of bee colony health across multiple years from the BIP/USDA dataset.

Before any modeling, we need to understand:
- What does each column mean?
- Are there any formatting issues or missing values?
- What does the distribution of our target variable look like?
- Does a linear relationship between colonies and winter loss actually exist?

### Why this question?

Honey bees pollinate roughly one-third of the food we eat. Winter colony loss is the primary indicator of bee population health — a high loss rate signals environmental stress, disease, or pesticide exposure. We ask whether the **number of colonies in a state** is a useful predictor of how much loss that state will experience. Do larger operations manage winter better, or does it make no difference?

In [ ]:
# Check column names and data types
print('Columns and data types:')
print(df.dtypes)

In [ ]:
# Check for missing values
print('Missing values per column:')
print(df.isnull().sum())

In [ ]:
# The numeric columns are stored as strings because of symbols like '%' and ','
# We need to strip those before converting to numbers

# Remove '%' from loss column and convert to float
df['Loss'] = df['Total Winter All Loss'].str.replace('%', '').str.strip()
df['Loss'] = pd.to_numeric(df['Loss'], errors='coerce')

# Remove commas from colony count and convert to float
df['Colonies_Clean'] = df['Colonies'].str.replace(',', '').str.strip()
df['Colonies_Clean'] = pd.to_numeric(df['Colonies_Clean'], errors='coerce')

# Drop rows where either column couldn't be parsed
df = df.dropna(subset=['Loss', 'Colonies_Clean'])

print('Rows remaining after cleaning:', len(df))
print()
print(df[['State', 'Colonies_Clean', 'Loss']].head(10))

In [ ]:
# Plot the distribution of winter loss across all state-year observations
plt.figure(figsize=(10, 5))
sns.histplot(df['Loss'], bins=30, color='goldenrod', edgecolor='black')
plt.axvline(x=df['Loss'].mean(), color='red', linestyle='--', linewidth=2,
            label=f'Mean loss ({df["Loss"].mean():.1f}%)')
plt.xlabel('Winter Colony Loss (%)', fontsize=13)
plt.ylabel('Number of Observations', fontsize=13)
plt.title('Distribution of Winter Bee Colony Loss Across U.S. States', fontsize=15)
plt.legend(fontsize=12)
plt.tight_layout()
plt.show()

print(f'Average winter loss: {df["Loss"].mean():.1f}%')
print(f'Median winter loss:  {df["Loss"].median():.1f}%')
print(f'Min: {df["Loss"].min():.1f}%   Max: {df["Loss"].max():.1f}%')

In [ ]:
# Scatterplot — is there any linear trend between colonies and winter loss?
# We should always look at the data before committing to a model!
plt.figure(figsize=(10, 6))
plt.scatter(df['Colonies_Clean'], df['Loss'],
            color='goldenrod', edgecolors='black', alpha=0.5,
            label='State-year observation')
plt.xlabel('Number of Colonies in State', fontsize=13)
plt.ylabel('Winter Colony Loss (%)', fontsize=13)
plt.title('Colonies vs. Winter Loss — Is There a Linear Trend?', fontsize=15)
plt.legend(fontsize=12)
plt.tight_layout()
plt.show()

## Step 3 — The Cost Function and Gradient Descent for Linear Regression

Now that we understand the data, let's talk about how the model actually learns.

### The Cost Function: Mean Squared Error

We need a way to measure how wrong our predictions are. We use the **Mean Squared Error (MSE)**:

$$C(w, b) = \frac{1}{2N}\sum_{i=1}^{N}\Big(\hat{y}^{(i)} - y^{(i)}\Big)^2$$

Squaring the error ensures that positive and negative mistakes don't cancel out, and penalizes large errors more heavily. Our goal is to find the values of $w$ and $b$ that make $C$ as small as possible.

### Deriving the Gradient (Chain Rule)

To apply gradient descent, we need the partial derivatives of $C$ with respect to $w$ and $b$. For a single training example $(x^{(i)}, y^{(i)})$ where $\hat{y}^{(i)} = wx^{(i)} + b$, we apply the chain rule:

$$\frac{\partial C}{\partial w} = (\hat{y}^{(i)} - y^{(i)}) \cdot x^{(i)}$$

$$\frac{\partial C}{\partial b} = (\hat{y}^{(i)} - y^{(i)})$$

### Two Flavors of Gradient Descent

**Flavor 1 — Batch Gradient Descent:** Compute the gradient over ALL training examples, then update once per epoch.

1. For each epoch **do:**
2. $\quad$ Compute $\frac{\partial C}{\partial w} = \frac{1}{N}\sum_{i=1}^{N}(\hat{y}^{(i)} - y^{(i)})x^{(i)}$
3. $\quad$ Compute $\frac{\partial C}{\partial b} = \frac{1}{N}\sum_{i=1}^{N}(\hat{y}^{(i)} - y^{(i)})$
4. $\quad w \leftarrow w - \alpha \frac{\partial C}{\partial w}$
5. $\quad b \leftarrow b - \alpha \frac{\partial C}{\partial b}$

This works but is **slow and memory-heavy** for large datasets.

**Flavor 2 — Stochastic Gradient Descent (SGD):** Update weights after **every single example**.

1. For each epoch **do:**
2. $\quad$ For $i = 1, \dots, N$ **do:**
3. $\quad\quad$ Compute $\hat{y}^{(i)} = wx^{(i)} + b$
4. $\quad\quad w \leftarrow w - \alpha(\hat{y}^{(i)} - y^{(i)})x^{(i)}$
5. $\quad\quad b \leftarrow b - \alpha(\hat{y}^{(i)} - y^{(i)})$

SGD is far more efficient in practice and is what our custom `LinearRegression` class implements.

## Step 4 — Preprocessing

Before training we need to:
1. **Select our feature and target** — pull out the two columns we care about
2. **Reshape X to 2D** — our model expects a table, even with one column
3. **Split into train and test sets** — so we can evaluate fairly
4. **Normalize the features** — colony counts reach into the hundreds of thousands; this scale difference causes gradient descent to overshoot

In [ ]:
# Step 1: Pull out feature (X) and target (y) as numpy arrays
X = df['Colonies_Clean'].values   # input — number of colonies
y = df['Loss'].values             # target — winter loss %

# Step 2: Reshape X from 1D (557,) to 2D (557, 1)
# Our LinearRegression class expects a 2D table as input
X = X.reshape(-1, 1)

print('X shape:', X.shape)
print('y shape:', y.shape)

In [ ]:
# Step 3: Split into train (80%) and test (20%) sets
np.random.seed(42)
indices = np.arange(len(X))
np.random.shuffle(indices)
split = int(0.8 * len(X))

X_train = X[indices[:split]]
X_test  = X[indices[split:]]
y_train = y[indices[:split]]
y_test  = y[indices[split:]]

print(f'Training set: {len(X_train)} observations')
print(f'Test set:     {len(X_test)} observations')

In [ ]:
# Step 4: Min-max normalize the features
# Formula: x_norm = (x - x_min) / (x_max - x_min)  →  maps everything to [0, 1]
#
# IMPORTANT: Always compute min/max from TRAINING data only.
# Apply those same values to the test data.
# Never let test data influence preprocessing — that leaks future information.

X_min = X_train.min()
X_max = X_train.max()

X_train_norm = (X_train - X_min) / (X_max - X_min)
X_test_norm  = (X_test  - X_min) / (X_max - X_min)

print('Before normalization — min:', X_train.min(), '| max:', X_train.max())
print('After normalization  — min:', round(X_train_norm.min(), 3), '| max:', round(X_train_norm.max(), 3))

## Step 5 — Training and Evaluation

Now we train our custom `LinearRegression` model and evaluate its performance. 

We will look at:
- The **loss curve** — does MSE decrease over epochs?
- The **regression line** plotted on top of the data
- **MSE and R²** scores on both train and test sets
- How different **learning rates** affect both the loss and the fitted line

In [ ]:
# Initialize and train our custom LinearRegression model
model = LinearRegression(learning_rate=0.01, epochs=50)
model.fit(X_train_norm, y_train)

print('Training complete!')
print(f'Learned weight (w): {model.weights[0]:.4f}')
print(f'Learned bias   (b): {model.bias:.4f}')

In [ ]:
# Create a smooth domain for drawing the regression line
domain_norm = np.linspace(0, 1, 200).reshape(-1, 1)
domain_original = domain_norm * (X_max - X_min) + X_min

# Side-by-side: regression line + loss curve
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Left: regression line on the training data
ax1.scatter(X_train, y_train, color='goldenrod', edgecolors='black', alpha=0.5, label='Training data')
ax1.plot(domain_original, model.predict(domain_norm), color='firebrick', linewidth=2.5, label='Regression line')
ax1.set_xlabel('Number of Colonies', fontsize=13)
ax1.set_ylabel('Winter Loss (%)', fontsize=13)
ax1.set_title('Regression Line on Training Data', fontsize=14)
ax1.legend(fontsize=11)

# Right: MSE loss over epochs
ax2.plot(range(1, len(model.loss_history) + 1), model.loss_history,
         marker='o', color='steelblue', linewidth=2, label='MSE')
ax2.set_xlabel('Epoch', fontsize=13)
ax2.set_ylabel('MSE', fontsize=13)
ax2.set_title('MSE at Each Epoch During Training', fontsize=14)
ax2.legend(fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# Evaluate on both train and test sets
# R² tells us what fraction of the target variance our model explains
# R² = 1.0 → perfect, R² = 0.0 → no better than predicting the mean, R² < 0 → worse than the mean

train_mse = model.mean_squared_error(X_train_norm, y_train)
train_r2  = model.r_squared(X_train_norm, y_train)
test_mse  = model.mean_squared_error(X_test_norm, y_test)
test_r2   = model.r_squared(X_test_norm, y_test)

print('=== Training Set ===')
print(f'  MSE: {train_mse:.4f}')
print(f'  R²:  {train_r2:.4f}')
print()
print('=== Test Set ===')
print(f'  MSE: {test_mse:.4f}')
print(f'  R²:  {test_r2:.4f}')

In [ ]:
# Experiment with different learning rates
# This time we show the REGRESSION LINE at each alpha — not just the loss curve
# This reveals how aggressively each learning rate moves the line toward the data

alphas = [0.001, 0.01, 0.05, 0.1]

fig, axs = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Effect of Learning Rate on the Fitted Regression Line', fontsize=16)

for ax, alpha in zip(axs.flat, alphas):
    m = LinearRegression(learning_rate=alpha, epochs=50)
    m.fit(X_train_norm, y_train)
    ax.scatter(X_train, y_train, color='goldenrod', edgecolors='black', alpha=0.4)
    ax.plot(domain_original, m.predict(domain_norm), color='firebrick', linewidth=2)
    ax.set_title(f'alpha = {alpha}', fontsize=13)
    ax.set_xlabel('Number of Colonies', fontsize=11)
    ax.set_ylabel('Winter Loss (%)', fontsize=11)

plt.tight_layout()
plt.show()

## Step 6 — Conclusion

### What did we find?

Our Linear Regression model was trained to predict the **winter colony loss percentage** of a U.S. state based on its **colony count**, using observations from the BIP/USDA dataset spanning 2010–2021.

### What do the results tell us?

The sign of the learned weight $w$ tells us the direction of the relationship — whether states with more colonies tend to lose a higher or lower percentage over winter. The R² score tells us how much of the variation in winter loss our model can explain from colony count alone.

The learning curve confirms that stochastic gradient descent is working: MSE decreases steadily across epochs, meaning the line is moving toward the true trend in the data with each pass.

### Why colony count as the predictor?

Colony count is a clean, consistently recorded variable across all states and years. The scale of a beekeeping operation could plausibly affect winter survival — through disease pressure, management quality, or resource availability. Even if the relationship is weak, understanding it informs conservation policy.

### Limitations of Linear Regression

1. **One feature only** — winter loss is almost certainly influenced by many variables: temperature, pesticide use, disease rates, beekeeper experience. A more complete model would include all of these.
2. **Assumes linearity** — if the true relationship between colonies and loss is non-linear, a straight line will systematically miss it.
3. **Sensitive to outliers** — very large states (California has 490,000+ colonies) can pull the regression line strongly. MSE squares the errors, making large mistakes especially costly.
4. **No uncertainty** — linear regression gives a point prediction with no measure of confidence. Future models will address this.

### What's next?

In the following notebooks we will explore **Logistic Regression** — which brings probability into classification problems — and eventually more powerful models that can capture non-linear relationships!